# Blackboard benchmark runs in Google Colab

The default run evaluates 20 official KramaBench CSV tasks sequentially using one local Qwen model and three personas. The tasks have numeric, text and list answers. Set `TASK_COUNT = 1` for a short check before starting the full batch.

The original three-task pilot is available as `kramabench`. A new run records its profile, task selection and recovery setting in Drive. MedAgentBench remains unavailable.


In [ ]:
#@title 1. Install the pinned project revision
readiness = None
import importlib
import importlib.metadata as metadata
import os
import re
import subprocess
import sys
from tempfile import TemporaryDirectory
from pathlib import Path

PROJECT_REVISION = "e98ec7669b6f7a44a7a8948a967666fb4ad74d13" #@param {type:"string"}
CHECKOUT_ROOT = Path("/content/intelligible-blackboard-checkouts")
LEGACY_SOURCE = Path("/content/intelligible-blackboard/src").resolve()
REPO_URL = "https://github.com/nikhil0744/intelligible-blackboard-architecture.git"
if not re.fullmatch(r"[0-9a-f]{40}", PROJECT_REVISION):
    raise ValueError("Use an immutable 40-character project commit, not a branch name.")
PROJECT_DIR = CHECKOUT_ROOT / PROJECT_REVISION
CHECKOUT_ROOT.mkdir(parents=True, exist_ok=True)
if not PROJECT_DIR.exists():
    with TemporaryDirectory(prefix="checkout-", dir=CHECKOUT_ROOT) as staging:
        checkout = Path(staging) / "project"
        subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(checkout)], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", PROJECT_REVISION], check=True)
        checkout.rename(PROJECT_DIR)
actual_revision = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True).strip()
if actual_revision != PROJECT_REVISION:
    raise ValueError(f"The checkout at {PROJECT_DIR} has unexpected revision {actual_revision}. Use an intact checkout.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR) + "[colab]"], check=True)
os.chdir(PROJECT_DIR)
for name in ("numpy", "pandas", "pydantic"):
    loaded = sys.modules.get(name)
    if loaded is not None and getattr(loaded, "__version__", None) != metadata.version(name):
        raise RuntimeError("A loaded package changed during installation. Restart the Colab session, then rerun the notebook.")
# Swap only source modules belonging to earlier notebook checkouts.
source_dir = (PROJECT_DIR / "src").resolve()
old_sources = {LEGACY_SOURCE}
old_sources.update(path.resolve() / "src" for path in CHECKOUT_ROOT.iterdir()
                   if path != PROJECT_DIR and re.fullmatch(r"[0-9a-f]{40}", path.name))
def from_previous_checkout(path):
    return bool(path) and any(Path(path).resolve().is_relative_to(old) for old in old_sources)

stale = [name for name, module in tuple(sys.modules.items())
         if from_previous_checkout(getattr(module, "__file__", None))]
if stale and "broker" in globals():
    broker.close()
for name in stale:
    del sys.modules[name]
sys.path[:] = [str(source_dir)] + [path for path in sys.path
                                  if path != str(source_dir) and not from_previous_checkout(path)]
importlib.invalidate_caches()
from benchmarks import batch as benchmark_batch
if not Path(benchmark_batch.__file__).resolve().is_relative_to(source_dir):
    raise RuntimeError("This session loaded a different benchmark package. Restart the Colab session, then rerun cell 1.")

print("Project revision:", actual_revision)
print("Python:", sys.version.split()[0])
print("Benchmark imports ready. Continue to cell 2.")

## Choose where to save and what to run

Drive is required for normal persistent runs. Temporary smoke mode explicitly uses disposable runtime storage. A **new run** creates a separate experiment each time; **resume** continues the exact saved selection/settings and skips committed results, including incorrect answers. Copy the printed run folder into `RESUME_RUN_DIR` after reconnecting.

To retest the same task after a code update, keep your GPU runtime and the revision specified in cell 1. Select **new run** with `TASK_COUNT = 1`. Cell 1 checks out that revision separately, so the old code, local model and Drive records stay available. Restart the session only if cell 1 reports a loaded dependency change. Resume requires the old run's code and configuration.

In a new run, select `kramabench20` for the 20-task profile or `kramabench` with `TASK_COUNT` at most 3 for the original pilot. To run specific tasks, set `TASK_IDS` to a comma-separated list from that profile instead of using the count. Cell 3 lists the selected tasks before cell 4 starts the batch. The batch time limit applies to each invocation.

For a resume check, set `STOP_AFTER_SAVED = 1`, select a batch with more than one task, reconnect in a new runtime, and resume that folder with `STOP_AFTER_SAVED = 0`. Resume loads the saved profile, task list, budgets and recovery setting. Keep the pinned code and original CPU/GPU mode.

In [ ]:
#@title 2. Experiment controls and Drive
readiness = None
from google.colab import drive
from benchmarks.batch import validate_destination

TEMPORARY_SMOKE = False #@param {type:"boolean"}
CPU_SMOKE = False #@param {type:"boolean"}
SAVE_ROOT = "/content/drive/MyDrive/blackboard_benchmarks" #@param {type:"string"}
BENCHMARK = "kramabench20" #@param ["kramabench20", "kramabench"]
RECOVERY_ENABLED = True #@param {type:"boolean"}
TASK_COUNT = 20 #@param [1, 2, 3, 5, 10, 20] {type:"raw"}
TASK_IDS = "" #@param {type:"string"}
BATCH_TIMEOUT_MINUTES = 180 #@param {type:"integer"}
VERBOSE_BATCH = False #@param {type:"boolean"}
SEED = 0 #@param {type:"integer"}
MODE = "new run" #@param ["new run", "resume"]
RESUME_RUN_DIR = "" #@param {type:"string"}
STOP_AFTER_SAVED = 0 #@param {type:"integer"}
RETRY_FAILED_ATTEMPTS = False #@param {type:"boolean"}

if TEMPORARY_SMOKE:
    ROOT = Path("/content/temporary-blackboard-benchmarks")
    print("Temporary smoke mode: download results before disconnecting; runtime storage can disappear.")
else:
    drive.mount("/content/drive")
    ROOT = Path(SAVE_ROOT)
validate_destination(ROOT, temporary=TEMPORARY_SMOKE)
CACHE_DIR, RESULTS_DIR = ROOT / "cache", ROOT / "runs"
if MODE == "resume" and not RESUME_RUN_DIR.strip():
    raise ValueError("Paste the saved run folder into RESUME_RUN_DIR.")
if STOP_AFTER_SAVED < 0:
    raise ValueError("STOP_AFTER_SAVED must be zero or a positive count.")
if MODE == "new run" and RETRY_FAILED_ATTEMPTS:
    raise ValueError("Retry failures applies only to resume mode.")
if MODE == "new run" and BATCH_TIMEOUT_MINUTES <= 0:
    raise ValueError("BATCH_TIMEOUT_MINUTES must be positive.")
print("Save root:", ROOT)
if MODE == "resume":
    print("Resume folder:", RESUME_RUN_DIR.strip())
elif TASK_IDS.strip():
    print("Requested task IDs:", TASK_IDS, "from", BENCHMARK)
else:
    print("Requested selection:", TASK_COUNT, "task(s) from", BENCHMARK)

In [ ]:
#@title 3. Prepare verified data, local Ollama and the pinned Qwen model
readiness = None
import json
from benchmarks.preparation import resources, prepare, PreparedBenchmark
from benchmarks.colab_setup import ensure_service
from benchmarks.batch import check_readiness, source_identity, package_identity, select_tasks
from benchmarks.persistence import RunStore
from llm_broker.config import InferenceSettings, build_broker

specifications = resources()
selected_profile = BENCHMARK
if MODE == "resume":
    previous_store = RunStore(Path(RESUME_RUN_DIR.strip()))
    saved = previous_store.manifest["configuration"]
    selected_profile = saved.get("benchmark", "kramabench")
    if saved["code_sha256"] != source_identity()["source_sha256"]:
        raise ValueError("Resume needs the original code revision from manifest.json; start a compatible runtime first.")
    if saved["packages"] != package_identity():
        raise ValueError("Resume needs the Python/package versions in manifest.json. Restore those versions or create a new run.")
    if saved["cpu_smoke"] != CPU_SMOKE:
        raise ValueError("Select the same CPU_SMOKE setting as the saved run.")
    settings = InferenceSettings(**saved["inference"])
else:
    settings = InferenceSettings(model=specifications["runtime"]["model"], ollama_host="http://127.0.0.1:11434")
profile_spec = specifications.get(selected_profile)
if profile_spec is None:
    raise ValueError(f"Unknown benchmark profile: {selected_profile}")
if not profile_spec.get("available", True):
    raise RuntimeError(profile_spec["reason"])
requested_ids = None
if MODE == "new run":
    if TASK_IDS.strip():
        requested_ids = [task_id.strip() for task_id in TASK_IDS.split(",")]
        if len(set(requested_ids)) != len(requested_ids) or not set(requested_ids) <= set(profile_spec["task_ids"]):
            valid = ", ".join(profile_spec["task_ids"])
            raise ValueError(f"Choose unique TASK_IDS from {selected_profile}: {valid}")
    elif not 1 <= int(TASK_COUNT) <= len(profile_spec["task_ids"]):
        raise ValueError(f"{selected_profile} has {len(profile_spec['task_ids'])} tasks. Change TASK_COUNT in cell 2.")
prepare(CACHE_DIR, selected_profile)
prepared = PreparedBenchmark.open(CACHE_DIR, selected_profile)
if MODE == "resume":
    selected_ids = [entry["task_id"] for entry in previous_store.selection]
else:
    selected_ids = [task.task_id for task in select_tasks(prepared, limit=int(TASK_COUNT), task_ids=requested_ids)]
print("Selected profile:", selected_profile)
print(f"Selected tasks ({len(selected_ids)}):", ", ".join(selected_ids))
print(json.dumps(prepared.coverage(), indent=2))
service = ensure_service(cpu_smoke=CPU_SMOKE)
if "broker" in globals():
    broker.close()
broker = build_broker(settings)
readiness = check_readiness(broker, settings, cpu_smoke=CPU_SMOKE)
print("Ready:", settings.model, "digest", readiness["model_digest"])
print("Hardware:", readiness["hardware"])
print("Package versions:", package_identity())
print("Recovery enabled:", saved["limits"].get("recovery_enabled", False) if MODE == "resume" else RECOVERY_ENABLED)
print("MedAgentBench:", specifications["medagentbench"]["reason"])

## Run and inspect the discussion

For the one-task retest, look for a successful CSV query, `evidence_ready`, a submitted answer, then posts from the reviewing agents. For a larger selection, the notebook prints the task's position and one saved outcome per attempt. Set `VERBOSE_BATCH = True` to print tool and discussion activity in larger runs. All events are saved either way. Cell 5 shows the independently graded answer and agent agreement separately.

Use the stop button to interrupt an executing task. Resume restarts that task with a new attempt ID and retains its partial records. Stop after an `attempt_saved` message to test resuming completed work. Storage failures stop the batch immediately.

In [ ]:
#@title 4. Execute the shared runner
import json
from benchmarks.batch import run_batch, resume_batch
from benchmarks.persistence import read_result
from benchmarks.runtime import Limits
if globals().get("readiness") is None:
    raise RuntimeError("Complete cell 3 successfully before running the benchmark.")

RUN_DIR = RESUME_RUN_DIR.strip() if MODE == "resume" else ""
saved_in_this_invocation = 0
task_positions = {task_id: index for index, task_id in enumerate(selected_ids, 1)}
detailed_logs = VERBOSE_BATCH or len(selected_ids) == 1

def show_progress(event):
    global RUN_DIR, saved_in_this_invocation
    kind = event["kind"]
    if kind == "run_created":
        RUN_DIR = event["run_dir"]
        print("SAVE THIS RUN FOLDER:", RUN_DIR, flush=True)
    elif kind == "session_started":
        task_id = event["task"]["task_id"]
        print(f"\nTask {task_positions[task_id]}/{len(selected_ids)} · {task_id}", flush=True)
        if detailed_logs:
            print(event["task"]["question"], flush=True)
    elif detailed_logs and kind == "contribution":
        post = event["contribution"]
        print(f"[{post['agent_id']}] {post['tag']} → {post['payload']['prediction']['claim']}", flush=True)
        print("  Why:", post["payload"]["explanation"]["rationale"][:500], flush=True)
        if post.get("metadata", {}).get("final_submission"):
            print("  Submitted answer:", post["metadata"]["final_output"], flush=True)
    elif detailed_logs and kind == "tool_requested":
        print(f"[{event['agent_id']}] tool {event['name']}: {json.dumps(event['arguments'])[:500]}", flush=True)
    elif detailed_logs and kind == "tool_observed":
        print("  Observation:", json.dumps(event["result"], ensure_ascii=False)[:700], flush=True)
    elif kind == "attempt_saved":
        RUN_DIR = event["run_dir"]
        saved_in_this_invocation += 1
        print("SAVED:", event["task_id"], event["status"], "grade:", event["grade"], flush=True)
        if event["status"] != "consensus" or event["grade"].get("status") != "scored":
            attempt_path = Path(RUN_DIR) / "attempts" / event["attempt_id"]
            result = read_result(attempt_path)
            failure = result.get("error") or {}
            repairs = failure.get("repair_reasons") or []
            detail = repairs[-1] if repairs else failure.get("error") or event["grade"].get("error")
            if detail:
                print("  Last failure detail:", detail[:500], flush=True)
            print("  Attempt records:", attempt_path, flush=True)
        if STOP_AFTER_SAVED and saved_in_this_invocation >= STOP_AFTER_SAVED:
            raise KeyboardInterrupt
    elif kind == "task_skipped":
        task_id = event["task_id"]
        print(f"SKIPPED: {task_positions[task_id]}/{len(selected_ids)} · {task_id} already saved", flush=True)
    elif kind == "run_time_limit":
        print(event["message"], flush=True)
    elif detailed_logs and (kind.startswith("recovery_") or kind in {"session_stopped", "deadlock_detected", "tool_request_blocked", "tools_closed", "query_scope_mismatch", "evidence_ready", "discussion_stalled"}):
        print(event, flush=True)

try:
    if MODE == "resume":
        store, summary = resume_batch(Path(RUN_DIR), prepared, broker, settings,
            cpu_smoke=CPU_SMOKE, retry_failures=RETRY_FAILED_ATTEMPTS,
            readiness=readiness, observer=show_progress)
    else:
        limits = Limits(recovery_enabled=RECOVERY_ENABLED, run_timeout_s=BATCH_TIMEOUT_MINUTES * 60)
        store, summary = run_batch(prepared, broker, settings, RESULTS_DIR, task_ids=selected_ids,
            seed=SEED, limits=limits, cpu_smoke=CPU_SMOKE, temporary=TEMPORARY_SMOKE,
            readiness=readiness, observer=show_progress)
    RUN_DIR = str(store.root)
except KeyboardInterrupt:
    print("Stopped. Saved attempts remain intact. Switch to resume mode to continue this run.")
print("Run folder:", RUN_DIR)

In [ ]:
#@title 5. Rebuild results and read a saved trace
import pandas as pd
from IPython.display import display
from benchmarks.persistence import RunStore
from benchmarks.presentation import trace_table, plot_summary
SHOW_LATEST_TRACE = False #@param {type:"boolean"}
PLOT_OUTCOMES = False #@param {type:"boolean"}

RUN_DIR = globals().get("RUN_DIR") or globals().get("RESUME_RUN_DIR", "").strip()
if not RUN_DIR:
    raise ValueError("Run an experiment first, or enter its saved folder in resume mode.")
with RunStore(Path(RUN_DIR)).writer() as stored:
    summary = stored.summarize()
print("Benchmark profile:", stored.manifest["configuration"].get("benchmark", "kramabench"))
counts = {k: summary[k] for k in ("selected", "started", "completed", "scored", "correct", "incorrect", "ungraded", "timed_out", "errored", "interrupted", "pending", "recovery_attempted_tasks", "recovery_confirmed_tasks")}
print(counts)
print("Success / all selected:", summary["success_rate_all_selected"], "denominator", summary["selected"])
print("Success / scored:", summary["success_rate_scored"], "denominator", summary["scored"])
print("Usage across all attempts (unknown stays null):", summary["usage_all_attempts"])
display(pd.DataFrame([{"task":r["task_id"], "status":r["status"], "consensus":r.get("consensus"),
    "answer":r.get("output"), "grade status":r.get("grade",{}).get("status"),
    "metric":r.get("grade",{}).get("metric"),
    "score":r.get("grade",{}).get("score"), "correct":r.get("grade",{}).get("correct"),
    "tool calls":r.get("tool_rounds"), "posts":r.get("turns"),
    "seconds":r.get("elapsed_s"), "attempt":r.get("attempt_id")} for r in summary["outcomes"]]))
if any(record.get("committed", True) for record in stored.attempts()):
    if summary["selected"] == 1 or SHOW_LATEST_TRACE:
        display(pd.DataFrame(trace_table(RUN_DIR)))
    if PLOT_OUTCOMES:
        print("Outcome figure:", plot_summary(RUN_DIR))
else:
    print("No committed attempt yet. Partial event/call logs remain in the run folder.")
print("Files:", RUN_DIR)

In [ ]:
#@title 6. Browse recorded tasks and download the run
import html
import shutil
from pathlib import Path
from google.colab import files
from IPython.display import HTML, display
from benchmarks.persistence import RunStore
from benchmarks.presentation import export_viewer

SHOW_PLAYBACK = True #@param {type:"boolean"}
DOWNLOAD_BUNDLE = False #@param {type:"boolean"}
run = Path(RUN_DIR)
if SHOW_PLAYBACK or DOWNLOAD_BUNDLE:
    records = [r for r in RunStore(run).attempts() if r.get("committed", True)]
    views = {}
    for record in records:
        attempt = record["attempt_id"]
        label = f"{record['task_id']} · {record['status']} · {attempt[:8]}"
        views[label] = export_viewer(run, attempt_id=attempt)
    if views:
        first = next(iter(views.values()))
        index = run / "presentation" / "index.html"
        options = "".join(
            f'<option value="{html.escape(path.relative_to(index.parent).as_posix(), quote=True)}">{html.escape(label)}</option>'
            for label, path in views.items()
        )
        index.write_text(
            '<!doctype html><meta charset="utf-8"><title>Recorded tasks</title>'
            '<style>body{font:16px system-ui;margin:12px}select{font:inherit}iframe{width:100%;height:90vh;border:0}</style>'
            '<label>Recorded task <select onchange="document.querySelector(\'iframe\').src=this.value">'
            + options + '</select></label><iframe title="Task playback" sandbox="allow-scripts" src="'
            + html.escape(first.relative_to(index.parent).as_posix(), quote=True) + '"></iframe>',
            encoding="utf-8",
        )
        if SHOW_PLAYBACK:
            import ipywidgets as widgets
            picker = widgets.Dropdown(options=list(views), description="Task:")
            playback = widgets.Output()
            def show_task(label):
                with playback:
                    playback.clear_output(wait=True)
                    embedded = html.escape(views[label].read_text(encoding="utf-8"), quote=True)
                    display(HTML('<iframe title="Task playback" sandbox="allow-scripts" style="width:100%;height:900px;border:0" srcdoc="' + embedded + '"></iframe>'))
            picker.observe(lambda change: show_task(change["new"]), names="value")
            display(picker, playback)
            show_task(picker.value)
    else:
        print("This run has no completed attempts to play.")
if DOWNLOAD_BUNDLE:
    archive = shutil.make_archive("/content/" + run.name, "zip", run)
    files.download(archive)


## Acceptance and stopping

Keep the run manifest and records as evidence. Verify one GPU task first. Use a small multi-task selection to check interruption and resume, then run the saved 20-task selection for the full profile. Check that saved grades, task denominators, contributions, tools and targets match the files in Drive. Incorrect answers can still demonstrate valid execution/grading; scripted unit checks cannot establish model performance.

To continue after reconnecting: rerun setup at the same revision, choose **resume**, paste the run folder, set the same CPU/GPU mode and leave `STOP_AFTER_SAVED = 0`. No keep-alive tricks, tunnels or external services are used. MedAgentBench feasibility remains pending.

In [ ]:
#@title Optional: stop this notebook's Ollama process
readiness = None
# This does not stop a pre-existing service owned by another process.
if "service" in globals():
    service.stop()
if "broker" in globals():
    broker.close()
print("Owned inference service stopped, if one was started. Drive results remain saved.")